In [1]:
import cv2
import numpy as np
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from tqdm import tqdm

# ---------------- CONFIG ----------------
ROOT = Path(
    "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
) / "SnowPole_Detection_Dataset"

LABEL_ROOT = ROOT / "labels"  # YOLO .txt files: labels/train/*.txt, labels/valid/*.txt

OUT_ROOT = Path("dataset_lda_pca/images")

MODALITIES = ["reflec", "signal", "nearir", "range"]  # channel indices 0,1,2,3
SPLITS = ["train", "valid", "test"]
IMG_EXTS = [".png", ".jpg", ".jpeg"]

FG_SAMPLES_PER_IMAGE = 500   # foreground pixels sampled per image for fitting
BG_SAMPLES_PER_IMAGE = 200   # background pixels sampled per image for fitting
MAX_PIXELS = 300_000


np.random.seed(42)
# ----------------------------------------


def read_first_channel(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Cannot read: {path}")
    if img.ndim == 3:
        img = img[:, :, 0]
    return img.astype(np.float32)


def load_stack(img_name: str, split: str) -> np.ndarray:
    """Returns H x W x 4 stack of all modalities."""
    return np.stack(
        [read_first_channel(ROOT / mod / split / img_name) for mod in MODALITIES],
        axis=-1
    )


def parse_yolo_labels(label_path: Path, H: int, W: int):
    boxes = []
    if not label_path.exists():
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, bw, bh = map(float, parts[:5])
            x1 = max(0, int((cx - bw / 2) * W))
            y1 = max(0, int((cy - bh / 2) * H))
            x2 = min(W, int((cx + bw / 2) * W))
            y2 = min(H, int((cy + bh / 2) * H))
            if x2 > x1 and y2 > y1:
                boxes.append((x1, y1, x2, y2))
    return boxes


def remove_lda_component(X: np.ndarray, direction: np.ndarray) -> np.ndarray:
    """Remove LDA direction from X via Gram-Schmidt."""
    return X - (X @ direction)[:, None] * direction[None, :]


# ============================================================
# STEP 1 — Collect foreground & background pixels (train only)
# ============================================================
print("=" * 60)
print("STEP 1: Collecting fg/bg pixels for LDA fitting...")
print("=" * 60)

train_ref_dir = ROOT / MODALITIES[0] / "train"
train_images = [p for p in train_ref_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

fg_pixels, bg_pixels = [], []
images_with_labels = 0

for img_path in tqdm(train_images, desc="Sampling"):
    stack = load_stack(img_path.name, "train")
    H, W, _ = stack.shape
    flat = stack.reshape(-1, 4)

    label_path = LABEL_ROOT / "train" / (img_path.stem + ".txt")
    boxes = parse_yolo_labels(label_path, H, W)

    if boxes:
        images_with_labels += 1
        fg_mask = np.zeros((H, W), dtype=bool)
        for x1, y1, x2, y2 in boxes:
            fg_mask[y1:y2, x1:x2] = True

        fg_flat = flat[fg_mask.reshape(-1)]
        bg_flat = flat[~fg_mask.reshape(-1)]

        if len(fg_flat) > 0:
            n = min(len(fg_flat), FG_SAMPLES_PER_IMAGE)
            fg_pixels.append(fg_flat[np.random.choice(len(fg_flat), n, replace=False)])

        n = min(len(bg_flat), BG_SAMPLES_PER_IMAGE)
        bg_pixels.append(bg_flat[np.random.choice(len(bg_flat), n, replace=False)])
    else:
        # No label: treat all pixels as background
        n = min(len(flat), BG_SAMPLES_PER_IMAGE)
        bg_pixels.append(flat[np.random.choice(len(flat), n, replace=False)])

fg_pixels = np.concatenate(fg_pixels)
bg_pixels = np.concatenate(bg_pixels)

print(f"  Images with labels : {images_with_labels}/{len(train_images)}")
print(f"  Foreground pixels  : {len(fg_pixels)}")
print(f"  Background pixels  : {len(bg_pixels)}")

# Cap to MAX_PIXELS
if len(fg_pixels) > MAX_PIXELS // 2:
    fg_pixels = fg_pixels[np.random.choice(len(fg_pixels), MAX_PIXELS // 2, replace=False)]
if len(bg_pixels) > MAX_PIXELS // 2:
    bg_pixels = bg_pixels[np.random.choice(len(bg_pixels), MAX_PIXELS // 2, replace=False)]

X_all = np.concatenate([fg_pixels, bg_pixels])
y_all = np.array([1] * len(fg_pixels) + [0] * len(bg_pixels), dtype=np.int32)
print(f"  Total for fitting  : {len(X_all)}")


# ============================================================
# STEP 2 — Fit LDA  (4 channels, binary class → 1 direction)
# ============================================================
print("\nSTEP 2: Fitting LDA...")

lda = LinearDiscriminantAnalysis(n_components=1)
lda.fit(X_all, y_all)

lda_dir = lda.scalings_[:, 0]
lda_dir /= (np.linalg.norm(lda_dir) + 1e-10)
print(f"  LDA direction (unit): {np.round(lda_dir, 4)}")


# ============================================================
# STEP 3 — Fit PCA on LDA-orthogonal residual (2 components)
# ============================================================
print("\nSTEP 3: Fitting PCA on residual...")

X_res = remove_lda_component(X_all, lda_dir)
pca2 = PCA(n_components=2)
pca2.fit(X_res)
print(f"  Explained variance : {np.round(pca2.explained_variance_ratio_, 4)}")
print(f"  Total captured     : {pca2.explained_variance_ratio_.sum():.4f}")


# ============================================================
# STEP 4 — Global normalization stats from train set
# ============================================================
print("\nSTEP 4: Computing global normalization stats...")

global_lo = np.full(3, np.inf)
global_hi = np.full(3, -np.inf)

for img_path in tqdm(train_images, desc="Stats"):
    stack = load_stack(img_path.name, "train")
    H, W, _ = stack.shape
    flat = stack.reshape(-1, 4)

    ch0 = (flat @ lda_dir).reshape(H, W)
    res = remove_lda_component(flat, lda_dir)
    pca_out = pca2.transform(res)
    ch1 = pca_out[:, 0].reshape(H, W)
    ch2 = pca_out[:, 1].reshape(H, W)

    for i, ch in enumerate([ch0, ch1, ch2]):
        global_lo[i] = min(global_lo[i], np.percentile(ch, 1))
        global_hi[i] = max(global_hi[i], np.percentile(ch, 99))

print(f"  Global lo: {np.round(global_lo, 4)}")
print(f"  Global hi: {np.round(global_hi, 4)}")

# Save for reproducibility / inference time use
np.save("lda_dir.npy",          lda_dir)
np.save("pca2_components.npy",  pca2.components_)
np.save("pca2_mean.npy",        pca2.mean_)
np.save("global_lo.npy",        global_lo)
np.save("global_hi.npy",        global_hi)
print("  Saved: lda_dir.npy | pca2_components.npy | pca2_mean.npy | global_lo.npy | global_hi.npy")


# ============================================================
# STEP 5 — Write fused images in original format
# ============================================================
def fuse(stack: np.ndarray) -> np.ndarray:
    """
    H x W x 4  →  H x W x 3  uint8
      R = LDA channel          (pole vs background discriminant)
      G = PCA residual comp 1  (next most informative direction)
      B = PCA residual comp 2
    """
    H, W, _ = stack.shape
    flat = stack.reshape(-1, 4)

    ch0 = (flat @ lda_dir).reshape(H, W)
    res = remove_lda_component(flat, lda_dir)
    pca_out = pca2.transform(res)
    ch1 = pca_out[:, 0].reshape(H, W)
    ch2 = pca_out[:, 1].reshape(H, W)

    def norm(ch, lo, hi):
        return ((np.clip(ch, lo, hi) - lo) / (hi - lo + 1e-6) * 255).astype(np.uint8)

    r = norm(ch0, global_lo[0], global_hi[0])
    g = norm(ch1, global_lo[1], global_hi[1])
    b = norm(ch2, global_lo[2], global_hi[2])

    return cv2.merge([b, g, r])  # OpenCV uses BGR internally


print("\nSTEP 5: Writing fused images...")

for split in SPLITS:
    out_dir = OUT_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    ref_dir = ROOT / MODALITIES[0] / split
    split_images = [p for p in ref_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

    for img_path in tqdm(split_images, desc=split):
        stack = load_stack(img_path.name, split)
        fused = fuse(stack)
        cv2.imwrite(str(out_dir / img_path.name), fused)  # saves in original format (.png/.jpg)

print(f"\n✅ Done. Fused images saved to: {OUT_ROOT}")

STEP 1: Collecting fg/bg pixels for LDA fitting...


Sampling: 100%|██████████| 1367/1367 [00:36<00:00, 37.08it/s]


  Images with labels : 1367/1367
  Foreground pixels  : 528248
  Background pixels  : 273400
  Total for fitting  : 300000

STEP 2: Fitting LDA...
  LDA direction (unit): [-0.0113  0.3317  0.3805 -0.8632]

STEP 3: Fitting PCA on residual...
  Explained variance : [0.6655 0.2473]
  Total captured     : 0.9128

STEP 4: Computing global normalization stats...


Stats: 100%|██████████| 1367/1367 [00:40<00:00, 33.44it/s]


  Global lo: [ -2235.7749 -34566.9648 -46068.4531]
  Global hi: [46670.6992 58322.3984 21655.5195]
  Saved: lda_dir.npy | pca2_components.npy | pca2_mean.npy | global_lo.npy | global_hi.npy

STEP 5: Writing fused images...


test: 100%|██████████| 197/197 [00:06<00:00, 30.37it/s]


✅ Done. Fused images saved to: dataset_lda_pca\images


In [ ]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="weighted_fusion.yaml",
    imgsz=1280,
    epochs=500,
    patience=80,        # early stopping
    batch=8,
    device=0,
    project="weighted_3pca",
    name="weighted_yolo11n_pca3",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=weighted_fusion.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D997ED2E50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [1]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="weighted_fusion.yaml",
    imgsz=1024,
    epochs=500,
    patience=80,        # early stopping
    batch=8,
    device=0,
    project="weighted_3pca",
    name="weighted_yolo11n_pca3",
    amp=False,
    augment=False,
    workers=0,
    box=12.0,    
    cls=0.3,     
    cos_lr=True,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=12.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.3, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=weighted_fusion.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000149009D2050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480